In [1]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l1, l2
from tensorflow.keras.initializers import GlorotUniform, HeNormal, RandomNormal, GlorotNormal, HeUniform
import tensorflow as tf
import random

SEED = 72
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

train_df = pd.read_csv('california_housing_train.csv')
test_df = pd.read_csv('california_housing_test.csv')

features = ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
            'total_bedrooms', 'population', 'households', 'median_income']
target = 'median_house_value'

x_train_full = train_df[features].values
y_train_full = train_df[target].values
x_test = test_df[features].values
y_test = test_df[target].values


mean = x_train_full.mean(axis=0)
std = x_train_full.std(axis=0)
x_train = (x_train_full - mean) / std
x_test = (x_test - mean) / std

2025-11-01 12:13:44.959131: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-01 12:13:44.959677: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-01 12:13:45.032783: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-01 12:13:46.696417: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur

In [2]:
def build_model(
    layers_config,
    activations,
    optimizer_name,
    learning_rate,
    dropout_rates,
    use_batch_norm,
    kernel_regularizer,
    bias_regularizer,
    initializer
):
    model = Sequential()
    for i, (units, act) in enumerate(zip(layers_config, activations)):
        if i == 0:
            model.add(Dense(units,
                            activation=act,
                            input_shape=(x_train.shape[1],),
                            kernel_initializer=initializer,
                            kernel_regularizer=kernel_regularizer,
                            bias_regularizer=bias_regularizer))
        else:
            model.add(Dense(units,
                            activation=act,
                            kernel_initializer=initializer,
                            kernel_regularizer=kernel_regularizer,
                            bias_regularizer=bias_regularizer))
        if use_batch_norm[i]:
            model.add(BatchNormalization())
        if dropout_rates[i] > 0:
            model.add(Dropout(dropout_rates[i]))
    model.add(Dense(1, activation='linear', kernel_initializer=initializer))
    if optimizer_name == 'adam':
        opt = Adam(learning_rate=learning_rate)
    elif optimizer_name == 'sgd':
        opt = SGD(learning_rate=learning_rate)
    elif optimizer_name == 'rmsprop':
        opt = RMSprop(learning_rate=learning_rate)
    else:
        raise ValueError("Unsupported optimizer")
    model.compile(optimizer=opt, loss='mse', metrics=['mae'])
    return model

In [3]:
import itertools

def random_architecture():
    n_layers = np.random.choice([2, 3, 4])
    start = np.random.choice([32, 64, 128, 256])
    layers = [start]
    for _ in range(1, n_layers):
        # Уменьшить до 8
        next_units = np.random.choice([u for u in [16, 32, 64, 128, 256] if u <= layers[-1]])
        layers.append(next_units)
    return layers

def random_activation(n):
    acts = []
    for _ in range(n):
        acts.append(np.random.choice(['relu', 'elu','tanh']))
    return acts

def random_dropout(n):
    return [np.random.choice([0.0, 0.1, 0.2]) for _ in range(n)]

def random_batch_norm(n):
    return [np.random.choice([True, False]) for _ in range(n)]

def random_regularizer():
    choice = np.random.choice(['none', 'l1', 'l2'])
    if choice == 'l1':
        return l1(1e-4)
    elif choice == 'l2':
        return l2(1e-4)
    else:
        return None
# надо связать с функцией активации 
def random_initializer():
    choice = np.random.choice(['glorot', 'he', 'normal','glorotnorm','henorm'])
    if choice == 'glorot':
        return GlorotUniform(seed=SEED)
    elif choice == 'he':
        return HeUniform(seed=SEED)
    elif choice == 'glorotnorm':
        return GlorotNormal(seed=SEED)
    elif choice == 'henorm':
        return HeNormal(seed=SEED)
    else:
        return RandomNormal(seed=SEED)

def random_optimizer():
    #return np.random.choice(['adam', 'sgd', 'rmsprop'])
    return np.random.choice(['adam','rmsprop','adamv'])

def random_lr():
    return np.random.choice([1e-4, 5e-4, 1e-3, 5e-3, 1e-2])

def random_batch_size():
    return np.random.choice([16, 32, 64, 128])

def random_epochs():
    return np.random.choice([200,300])

In [4]:
results = []
N_EXPERIMENTS = 500
INTERVAL = 25 

early_stop = EarlyStopping(monitor='val_mae', patience=50, restore_best_weights=True)

for exp_id in range(N_EXPERIMENTS):
    print(f"Эксперимент {exp_id + 1}/{N_EXPERIMENTS}")
    
    # Генерация
    layers = random_architecture()
    n = len(layers)
    activations = random_activation(n)
    dropout_rates = random_dropout(n)
    batch_norm_flags = random_batch_norm(n)
    kernel_reg = random_regularizer()
    bias_reg = random_regularizer()
    initializer = random_initializer()
    opt_name = random_optimizer()
    lr = random_lr()
    batch_size = random_batch_size()
    max_epochs = random_epochs()
    

    try:
        model = build_model(
            layers_config=layers,
            activations=activations,
            optimizer_name=opt_name,
            learning_rate=lr,
            dropout_rates=dropout_rates,
            use_batch_norm=batch_norm_flags,
            kernel_regularizer=kernel_reg,
            bias_regularizer=bias_reg,
            initializer=initializer
        )
    except Exception as e:
        print(f"Ошибка при создании модели: {e}")
        continue

    history = model.fit(
        x_train_full, y_train_full,
        validation_split=0.2,
        epochs=max_epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=0
    )
    
    actual_epochs = len(history.history['mae'])
    
    train_mae = history.history['mae'][-1]
    val_mae = history.history['val_mae'][-1]
    test_mae = model.evaluate(x_test, y_test, verbose=1)[1]
    
    # Сохранение результата
    results.append({
        'ID': exp_id + 1,
        'layers': str(layers),
        'activations': str(activations),
        'optimizer': opt_name,
        'learning_rate': lr,
        'batch_size': batch_size,
        'epochs': actual_epochs,
        'dropout': str(dropout_rates),
        'batch_norm': str(batch_norm_flags),
        'kernel_regularizer': str(kernel_reg),
        'bias_regularizer': str(bias_reg),
        'initializer': str(initializer.__class__.__name__),
        'train_mae': train_mae,
        'val_mae': val_mae,
        'test_mae': test_mae
    })
    
    # Сохраняем лучшую модель
    if exp_id == 0 or (test_mae < best_test_mae and test_mae>0):
        best_test_mae = test_mae
        best_model = model
        best_model.save("best_model.keras")

    # Промежуточная запись каждые INTERVAL экспериментов
    if (exp_id + 1) % INTERVAL == 0:
        df_partial = pd.DataFrame(results)
        df_partial.to_csv("experiment_results2.csv", index=False)
        print(f"Промежуточные результаты (до эксперимента {exp_id + 1}) сохранены в experiment_results.csv")

Эксперимент 1/500


2025-11-01 12:13:47.962205: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/home/user1/repos/data_science/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 51812098048.0000 - mae: 201043.4062
Эксперимент 2/500
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: nan - mae: nan
Эксперимент 3/500
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 8168229376.0000 - mae: 69432.0234
Эксперимент 4/500
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 349282900929681593439617024.0000 - mae: 18325184708608.0000
Эксперимент 5/500
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 55164653568.0000 - mae: 205846.7344
Эксперимент 6/500
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 55163273216.0000 - mae: 205843.2812
Эксперимент 7/500
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 53473456128.0000 - mae: 201868.5469
Эксперимент 8/500
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: nan - mae: nan
Эксперимент 9/500
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: nan - mae: nan
Эксперимент 10/500
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 15370594304.0000 - mae: 110147.8047
Эксперимент 11/500
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('experiment_results2.csv', on_bad_lines='skip')

df['test_mae'] = pd.to_numeric(df['test_mae'], errors='coerce')

valid = df[np.isfinite(df['test_mae'])]

top_10 = valid.nsmallest(10, 'test_mae')

result = top_10[['ID', 'test_mae', 'layers', 'activations', 'optimizer', 'learning_rate', 'batch_size', 'epochs', 'dropout', 'batch_norm']]

result